In [ ]:
pip install websocket-client

In [ ]:
import websocket
import json
import random
import string
import threading
import time
import pandas as pd
from IPython.display import clear_output
from datetime import datetime

# 1. إعدادات الاتصال
WS_URL = "wss://data.tradingview.com/socket.io/websocket"
HEADERS = {
    "Origin": "https://data.tradingview.com",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

class TVConnection:
    def __init__(self, conn_id, symbols):
        self.conn_id = conn_id
        self.symbols = symbols
        self.received_count = 0
        self.is_connected = False
        self.last_event = "Initializing..."
        self.live_prices = {} 
        self.reconnect_count = 0

    def log_event(self, msg):
        timestamp = datetime.now().strftime("%H:%M:%S")
        self.last_event = f"[{timestamp}] {msg}"

    def format_message(self, func, params):
        payload = json.dumps({"m": func, "p": params}, separators=(",", ":"))
        return f"~m~{len(payload)}~m~{payload}"

    def run(self):
        while True: 
            quote_session = f"qs_{''.join(random.choice(string.ascii_lowercase) for _ in range(12))}"
            
            def on_open(ws):
                self.is_connected = True
                self.log_event("✅ Connected. Subscribing...")
                ws.send(self.format_message("set_auth_token", ["unauthorized_user_token"]))
                ws.send(self.format_message("quote_create_session", [quote_session]))
                ws.send(self.format_message("quote_set_fields", [quote_session, "lp"]))
                for s in self.symbols:
                    ws.send(self.format_message("quote_add_symbols", [quote_session, s]))

            def on_message(ws, msg):
                if msg.startswith("~h~"):
                    ws.send(msg)
                    return
                if "~m~" in msg:
                    for p in msg.split("~m~"):
                        if not p or p.isdigit(): continue
                        try:
                            data = json.loads(p)
                            if data.get("m") == "qsd":
                                quote_data = data['p'][1]
                                sym = quote_data.get('n')
                                price = quote_data.get('v', {}).get('lp')
                                if sym and price:
                                    self.live_prices[sym] = price
                                    self.received_count += 1
                        except: pass

            def on_error(ws, err):
                self.is_connected = False

            def on_close(ws, code, msg):
                self.is_connected = False

            ws = websocket.WebSocketApp(WS_URL, header=HEADERS, on_open=on_open, on_message=on_message, on_error=on_error, on_close=on_close)
            ws.run_forever()
            
            self.reconnect_count += 1
            time.sleep(5) 

# 2. تشغيل المراقبة
try:
    df = pd.read_csv("symbols.csv")
    all_symbols = df["Symbol"].tolist()
    
    connections = []
    for i in range(10): 
        start, end = i*100, (i+1)*100
        conn = TVConnection(i + 1, all_symbols[start:end])
        connections.append(conn)
        threading.Thread(target=conn.run, daemon=True).start()
        time.sleep(0.5)

    start_time = time.time()
    while True:
        clear_output(wait=True)
        print(f"🕵️ TradingView LIVE Monitor | Running: {int(time.time()-start_time)}s")
        print("=" * 100)
        print(f"{'ID':<4} | {'Status':<15} | {'Updates':<8} | {'Last Price Sample (Live)'}")
        print("-" * 100)
        
        for c in connections:
            status = "🟢 ACTIVE" if c.is_connected else "🔄 RECONNECTING"
            sample_price = ""
            if c.live_prices:
                random_sym = random.choice(list(c.live_prices.keys()))
                sample_price = f"{random_sym}: {c.live_prices[random_sym]}"
            
            print(f"{c.conn_id:<4} | {status:<15} | {c.received_count:<8} | {sample_price}")
            
        print("-" * 100)
        print(f"💡 Reconnect Attempts: {sum(c.reconnect_count for c in connections)}")
        print("To stop: Click the STOP button in Jupyter.")
        time.sleep(1)
except KeyboardInterrupt:
    print("Stopped.")
except Exception as e:
    print(f"Error: {e}")
